# Phase 5.1 — Frozen expert retrieval evaluation

## Goal

Turn the technically verified Phase 5 index into a defensible expert gold set and a preregistered BM25/dense/hybrid comparison.

Run this same notebook from top to bottom at every checkpoint. It automatically advances only when the current human input is complete:

1. create and validate 50-query intake;
2. create two blinded independent-labeler packets;
3. create adjudication and citation-audit packet;
4. freeze labels, calculate metrics and confidence intervals, and prepare the supervisor decision packet.

The notebook never invents queries or labels, never modifies the original source or the signed Phase 4/5 outputs, never calls an external LLM or embedding API, and never promotes a production model automatically.


## Setup

### 1. Mount Drive and locate the clean project

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import zipfile

PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PACKAGE_FILENAME = "PHASE_5_1_EXPERT_EVALUATION_PACKAGE.zip"
EXPECTED_PACKAGE_SHA256 = "42c87ff549d8e64a4d461f600dea3301326ad9fe3964dbefff2e4ab77f2be8ea"

my_drive = Path('/content/drive/MyDrive')
candidates = [
    path.parent.parent
    for path in my_drive.rglob('config/project.yaml')
    if path.parent.parent.name == PROJECT_FOLDER_NAME
]
candidates = sorted(set(path.resolve() for path in candidates))
assert len(candidates) == 1, f"Expected exactly one clean project; found: {candidates}"
PROJECT_ROOT = candidates[0]
PACKAGE_PATH = PROJECT_ROOT / PACKAGE_FILENAME
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
assert (PROJECT_ROOT / 'data' / 'indexes').exists(), "Phase 5 index folder is missing"
print(f"Project root: {PROJECT_ROOT}")


### 2. Verify and safely install the signed Phase 5.1 extension

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

actual_package_sha = file_sha256(PACKAGE_PATH)
assert actual_package_sha == EXPECTED_PACKAGE_SHA256, (
    f"Phase 5.1 package hash mismatch: {actual_package_sha}"
)

with zipfile.ZipFile(PACKAGE_PATH) as archive:
    members = archive.infolist()
    for member in members:
        target = (PROJECT_ROOT / member.filename).resolve()
        assert str(target).startswith(str(PROJECT_ROOT.resolve()) + os.sep), (
            f"Unsafe archive path: {member.filename}"
        )
    package_manifest = json.loads(
        archive.read('PHASE_5_1_PACKAGE_MANIFEST.json').decode('utf-8')
    )
    assert package_manifest['pipeline_version'] == 'phase5_1_expert_evaluation_v1'
    assert package_manifest['package_mode'] == 'ADDITIVE_EXTENSION'
    assert package_manifest['automatic_production_promotion'] is False
    assert package_manifest['anti_leakage_policy'] == 'FROZEN_TEST_ONLY_NO_TUNING'
    for entry in package_manifest['files']:
        packaged_bytes = archive.read(entry['path'])
        assert hashlib.sha256(packaged_bytes).hexdigest() == entry['sha256']
    for member in members:
        if member.is_dir():
            continue
        destination = PROJECT_ROOT / member.filename
        packaged_hash = hashlib.sha256(archive.read(member.filename)).hexdigest()
        if destination.exists():
            assert file_sha256(destination) == packaged_hash, (
                f"Refusing to overwrite a changed project file: {member.filename}"
            )
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(archive.read(member.filename))

print(f"Verified package SHA-256: {actual_package_sha}")
print("Phase 5.1 additive extension installed safely.")


### 3. Install dependencies and run the complete project test suite

In [ ]:
requirements = [
    PROJECT_ROOT / 'requirements' / 'phase5.txt',
    PROJECT_ROOT / 'requirements' / 'phase5_1.txt',
]
print('Installing/checking controlled retrieval-evaluation packages...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q']
    + [item for path in requirements for item in ('-r', str(path))],
    check=True,
)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Running the complete foundation-through-expert-evaluation test suite...')
tests = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(PROJECT_ROOT / 'tests')],
    text=True,
    capture_output=True,
)
print(tests.stdout[-6000:])
if tests.stderr:
    print(tests.stderr[-3000:])
assert tests.returncode == 0, 'Tests failed; Phase 5.1 did not start.'
print('Complete project test suite passed.')


## Steps

### 4. Advance the controlled evaluation state machine

The first run is intentionally quick: it creates the query-intake workbook and stops without downloading model weights. Later runs advance only after the required human workbook is complete.


In [ ]:
from devoteam_reference_ai.phase5_1_evaluation import advance_phase5_1

CONFIG_PATH = PROJECT_ROOT / 'config' / 'phase5_1_evaluation.yaml'
result = advance_phase5_1(PROJECT_ROOT, CONFIG_PATH, progress=print)
print(json.dumps(result, indent=2, ensure_ascii=False, default=str))


## Checks

### 5. Verify state and print the exact next action

In [ ]:
from devoteam_reference_ai.phase5_1_evaluation import verify_phase5_1

RUN_ROOT = (
    PROJECT_ROOT
    / 'data'
    / 'evaluations'
    / '20260714T154731Z_129ff982c8'
    / 'phase5_1_expert_evaluation_v1'
)
verified = verify_phase5_1(RUN_ROOT)
status = verified['status']
print(f"PHASE 5.1 STATUS: {status}")
print(f"Output: {RUN_ROOT}")

if status == 'AWAITING_QUERY_INTAKE':
    print('NEXT: Open human_inputs/PHASE_5_1_QUERY_INTAKE.xlsx.')
    print('Complete Governance and all 50 Queries rows, save it, then rerun this notebook from the top.')
elif status == 'AWAITING_INDEPENDENT_LABELS':
    print('NEXT: Two different Devoteam experts independently complete LABELER_1.xlsx and LABELER_2.xlsx.')
    print('They must not discuss labels before both packets are finished. Then rerun this notebook.')
elif status == 'AWAITING_ADJUDICATION_AND_CITATION_AUDIT':
    print('NEXT: The independent adjudicator completes ADJUDICATION_AND_CITATION_AUDIT.xlsx, then rerun.')
elif status == 'EXPERT_EVALUATION_COMPLETE':
    print(f"Decision status: {verified['decision_status']}")
    print(f"Recommended baseline for supervisor review: {verified.get('recommended_candidate')}")
    print('Automatic production promotion remains prohibited; send the final report for senior review.')
else:
    raise AssertionError(f"Unexpected Phase 5.1 state: {status}")


## Next steps

- Do not fill labels yourself unless you are one of the formally assigned domain experts.
- Do not create queries from the reference spreadsheet, retrieved files, or bootstrap probes.
- Do not tune BM25, dense, hybrid weights, or a reranker using this frozen test set.
- Send the final printed block and report for supervisor review when the state becomes `EXPERT_EVALUATION_COMPLETE`.
